# 06. RAGAS

Оцениваем RAG-ответы стандартными метриками RAGAS на тех же 30 вопросах.

Метрики:

- **faithfulness** — подтверждаются ли утверждения ответа найденным контекстом;
- **answer relevancy** — насколько ответ относится к вопросу;
- **context precision** — стоят ли релевантные чанки выше в top-5;
- **context recall** — покрывает ли найденный контекст эталонный ответ;
- **answer correctness** — насколько ответ близок к эталону.

Все метрики RAGAS — от 0 до 1. Судья — DeepSeek, эмбеддинги — e5-small.

Это дополнение к кастомному judge из ноутбуков 04–05: RAGAS даёт
стандартизированные retrieval- и answer-метрики, но требует больше вызовов LLM.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

repo: D:\diffeq-wiki-rag


In [2]:
import json
import time

import pandas as pd

from app.rag.pipeline import build_rag_pipeline
from app.rag.store import build_vectorstore
from app.settings import get_settings
from eval.metrics import load_golden
from eval.ragas_eval import METRIC_NAMES, evaluate_ragas

items = load_golden()
settings = get_settings()
store = build_vectorstore()
pipeline = build_rag_pipeline(
    retriever=store.as_retriever(search_kwargs={"k": settings.top_k})
)

print("вопросов:", len(items))

D:\diffeq-wiki-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5477.71it/s]

вопросов: 30


## 1. Ответы и контекст

Для каждого вопроса:

1. retrieval top-5;
2. генерация RAG-ответа DeepSeek;
3. сохранение контекста — RAGAS оценивает каждый чанк отдельно.

30 ответов DeepSeek, затем один общий прогон RAGAS.

In [3]:
rows = []
started = time.perf_counter()

for index, item in enumerate(items, start=1):
    result = pipeline.run(item["question"])
    rows.append(
        {
            "id": item["id"],
            "question": item["question"],
            "answer": result.answer,
            "contexts": [document.page_content for document in result.documents],
            "reference": item["reference_answer"],
        }
    )
    print(
        f"[{index}/{len(items)}] {item['id']} | "
        f"contexts {len(result.documents)} | "
        f"elapsed {time.perf_counter() - started:.0f}s"
    )

print("готово к RAGAS")

[1/30] lipschitz-def-01 | contexts 5 | elapsed 3s


[2/30] lipschitz-example-02 | contexts 5 | elapsed 6s


[3/30] lipschitz-props-03 | contexts 5 | elapsed 11s


[4/30] lipschitz-picard-04 | contexts 5 | elapsed 14s


[5/30] linearode-general-05 | contexts 5 | elapsed 16s


[6/30] linearode-homconst-06 | contexts 5 | elapsed 18s


... (20 outputs left out)

[27/30] multi-wronskian-ode-27 | contexts 5 | elapsed 78s


[28/30] multi-integrating-linear-28 | contexts 5 | elapsed 81s


[29/30] multi-charpoly-basis-29 | contexts 5 | elapsed 84s


[30/30] multi-reduction-system-30 | contexts 5 | elapsed 87s
готово к RAGAS


## 2. RAGAS

Один вызов `evaluate` запускает пять метрик. Каждая метрика делает несколько
LLM-вызовов, поэтому шаг идёт 10–15 минут.

`max_workers=3` — компромисс между скоростью и лимитами DeepSeek.
`AnswerRelevancy.strictness=1`, потому что DeepSeek не поддерживает `n > 1`.

In [4]:
ragas_started = time.perf_counter()
ragas_result = evaluate_ragas(rows, max_workers=3)
print(f"RAGAS elapsed: {time.perf_counter() - ragas_started:.0f}s")

summary_df = pd.DataFrame([ragas_result["summary"]])
summary_df

RAGAS elapsed: 555s


,questions,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness,failed
0,30,0.8926,0.782,0.5782,0.675,0.5203,0


## 3. По темам

In [5]:
topic_by_id = {item["id"]: item["topic"] for item in items}
scored = pd.DataFrame(ragas_result["rows"])
scored["topic"] = scored["id"].map(topic_by_id)

topic_df = (
    scored.groupby("topic")[list(METRIC_NAMES)]
    .mean()
    .round(3)
    .reset_index()
)
topic_df["n"] = scored.groupby("topic").size().values
topic_df

,topic,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness,n
0,integrating_factor,0.981,0.859,0.917,1.000,0.807,3
1,linear_ode,0.920,0.945,0.572,0.633,0.536,5
2,lipschitz,0.865,0.955,0.275,0.438,0.423,4
3,multi_hop,0.894,0.765,0.720,0.800,0.483,5
4,ode,0.842,0.685,0.493,0.625,0.477,4
5,picard,0.802,0.377,0.440,0.433,0.467,5
6,wronskian,0.982,0.971,0.717,0.917,0.539,4


## 4. По вопросам

In [6]:
per_question = scored[["id", "topic", *METRIC_NAMES]].copy()
per_question

,id,topic,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
0,lipschitz-def-01,lipschitz,0.8462,0.9425,0.0000,0.7500,0.3950
1,lipschitz-example-02,lipschitz,0.8095,0.8984,0.2000,0.6667,0.3853
2,lipschitz-props-03,lipschitz,0.9524,0.9968,0.2000,0.3333,0.3872
3,lipschitz-picard-04,lipschitz,0.8500,0.9814,0.7000,0.0000,0.5263
4,linearode-general-05,linear_ode,1.0000,1.0000,1.0000,1.0000,0.5922
5,linearode-homconst-06,linear_ode,0.9091,0.8532,0.3333,0.0000,0.3075
6,linearode-charpoly-07,linear_ode,0.8333,0.9433,0.3250,0.6667,0.4652
7,linearode-nonhom-08,linear_ode,1.0000,1.0000,0.2000,0.5000,0.6584
8,linearode-system-09,linear_ode,0.8571,0.9286,1.0000,1.0000,0.6579
9,ode-def-10,ode,0.8667,0.0000,0.0000,0.0000,0.2242


## 5. Сохранение

In [7]:
payload = {
    "summary": ragas_result["summary"],
    "by_topic": topic_df.to_dict(orient="records"),
    "rows": ragas_result["rows"],
}
output = Path("eval/results/ragas_experiment.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("сохранено:", output)

сохранено: eval\results\ragas_experiment.json


## 6. Как читать

| Метрика | Что означает |
|---|---|
| faithfulness | все ли утверждения ответа подтверждаются контекстом |
| answer relevancy | отвечает ли ответ на заданный вопрос |
| context precision | релевантные чанки стоят выше нерелевантных |
| context recall | хватает ли контекста, чтобы покрыть эталон |
| answer correctness | близость ответа к эталону |

Если faithfulness высокая, а context recall низкая — ответ не выдумывает,
но retrieval не находит всё необходимое. Если context precision низкая —
в top-5 много шума.

RAGAS строже кастомного judge: это нормализованные 0–1, а не 1–5.

## 7. Выводы

### Сводка (0–1)

| Метрика | Значение |
|---|---|
| faithfulness | **0.893** |
| answer relevancy | 0.782 |
| context precision | 0.578 |
| context recall | 0.675 |
| answer correctness | 0.520 |

30 из 30 вопросов оценены, ошибок — 0. RAGAS занял ~555 секунд,
генерация ответов — ~87 секунд.

### По темам

| Тема | n | faithfulness | answer relevancy | context precision | context recall | answer correctness |
|---|---|---|---|---|---|---|
| integrating factor | 3 | 0.981 | 0.859 | 0.917 | 1.000 | **0.807** |
| linear ODE | 5 | 0.920 | 0.945 | 0.572 | 0.633 | 0.536 |
| lipschitz | 4 | 0.865 | 0.955 | 0.275 | 0.438 | 0.423 |
| multi-hop | 5 | 0.894 | 0.765 | 0.720 | 0.800 | 0.483 |
| ODE | 4 | 0.842 | 0.685 | 0.493 | 0.625 | 0.477 |
| Picard | 5 | 0.802 | 0.377 | 0.440 | 0.433 | 0.467 |
| Wronskian | 4 | 0.982 | 0.971 | 0.717 | 0.917 | 0.539 |

### Что видно

- **Faithfulness 0.893** — ответы почти не выдумывают. Основная проблема
  не в галлюцинациях, а в retrieval.
- **Context precision 0.578** и **context recall 0.675** — в top-5 много
  шума, а часть нужных фактов не находится. Хуже всего Lipschitz (0.275 / 0.438)
  и Picard (0.440 / 0.433).
- **Answer relevancy 0.782**, но у пяти вопросов ровно 0.0. Это не сбой:
  модель честно отвечает «в контексте этого нет» — например, для
  `picard-theorem-14`, `picard-nonunique-16` и `multi-reduction-system-30`.
  То есть retrieval не принёс нужный раздел.
- **Answer correctness 0.520** — RAGAS строже кастомного judge: сравнивает
  ответ с эталоном семантически и по фактам. Лучший результат — integrating
  factor (0.807), худший — lipschitz (0.423).

### Вывод

RAGAS подтверждает главный вывод проекта: **узкое место — retrieval, а не
генерация**. Ответы заземлены, но top-5 не всегда содержит нужные факты.

Следующие шаги:

1. reranking или гибридный поиск;
2. увеличить top-k и проверить влияние на context recall;
3. для multi-hop и Picard — маршрутизация в агент, который делает несколько
   поисков;
4. прогнать RAGAS на ответах агента и сравнить с RAG.
